In [1]:
# 1. GloVe 임베딩 다운로드 (최초 1회 실행)
!wget https://huggingface.co/stanfordnlp/glove/resolve/main/glove.6B.zip
!unzip -q glove.6B.zip


--2026-11-06 22:26:17--  https://huggingface.co/stanfordnlp/glove/resolve/main/glove.6B.zip
Resolving huggingface.co (huggingface.co)... 99.86.101.39, 99.86.101.56, 99.86.101.36, ...
Connecting to huggingface.co (huggingface.co)|99.86.101.39|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/621ffdc136468d709f18095d/ded22944ac944002f423d1cdb83de044851330e1a7fe19d74dc59ffa632ad294?X-Xet-Cas-Uid=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27glove.6B.zip%3B+filename%3D%22glove.6B.zip%22%3B&xip=VnnuIVYP06M&response-content-type=application%2Fzip&user_id=public&Expires=1794007577&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjIxZmZkYzEzNjQ2OGQ3MDlmMTgwOTVkL2RlZDIyOTQ0YWM5NDQwMDJmNDIzZDFjZGI4M2RlMDQ0ODUxMzMwZTFhN2ZlMTlkNzRkYzU5ZmZhNjMyYWQyOTRcXD9YLVhldC1DYXMtVWlkPXB1YmxpYyZyZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSomeGlwPVZubnVJVllQMDZNJnJlc3BvbnNlLWNvbnRlb

In [2]:
# 2. 라이브러리 로드 및 데이터 준비
import tensorflow as tf
import numpy as np
import re
import matplotlib.pyplot as plt
from datasets import load_dataset
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import layers, models

# IMDB 데이터 로드
imdb = load_dataset("stanfordnlp/imdb")   # IMDB 영화 리뷰 감성 데이터셋 (학습용·테스트용 각 25,000개)

# 텍스트 정제 함수 (소문자화, 특수문자 제거 등)
def clean_text(text):
    text = text.lower()                                # 모두 소문자로 변환
    text = re.sub(r"<br\s*/?>", " ", text)             # HTML 줄바꿈 태그 제거
    text = re.sub(r"[^a-z0-9']", " ", text)            # 알파벳/숫자/' 를 제외한 문자 제거
    return text

# 정제된 텍스트와 레이블 추출
train_texts = [clean_text(x) for x in imdb['train']['text']]
test_texts  = [clean_text(x) for x in imdb['test']['text']]
train_labels = list(imdb['train']['label'])
test_labels  = list(imdb['test']['label'])

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/unsupervised-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 42.0MB            

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [3]:
# 3. 토큰화 및 시퀀스 패딩
VOCAB_SIZE = 20000              # 단어 사전 크기 제한 (자주 등장하는 20,000단어 사용)
MAX_LEN = 200                   # 각 문장을 최대 200단어까지 자르고 패딩

# 토크나이저로 단어 인덱싱
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")  # OOV: 사전에 없는 단어 처리용
tokenizer.fit_on_texts(train_texts)

# 텍스트를 정수 시퀀스로 변환
train_seq = tokenizer.texts_to_sequences(train_texts)
test_seq  = tokenizer.texts_to_sequences(test_texts)

# 시퀀스를 길이 200으로 패딩 (post: 뒤쪽에 0을 채움)
train_pad = pad_sequences(train_seq, maxlen=MAX_LEN, padding='post')
test_pad  = pad_sequences(test_seq, maxlen=MAX_LEN, padding='post')

# 넘파이 배열로 변환 (모델 입력용)
train_pad = np.array(train_pad)
test_pad = np.array(test_pad)
train_labels = np.array(train_labels)
test_labels = np.array(test_labels)

In [4]:
# 4. GloVe 임베딩 불러오기 (100차원)
embedding_index = {}
with open("glove.6B.100d.txt", encoding="utf8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype='float32')
        embedding_index[word] = vector

# 임베딩 매트릭스 생성
embedding_dim = 100
word_index = tokenizer.word_index
num_words = min(VOCAB_SIZE, len(word_index) + 1)

embedding_matrix = np.zeros((num_words, embedding_dim))
for word, i in word_index.items():
    if i < num_words and word in embedding_index:
        embedding_matrix[i] = embedding_index[word]


In [5]:
# 5. 모델 정의 함수
def build_model(trainable=False):
    model = models.Sequential()

    # 사전학습된 GloVe 임베딩 사용
    model.add(layers.Embedding(input_dim=num_words,
                               output_dim=embedding_dim,
                               weights=[embedding_matrix],
                               input_length=MAX_LEN,
                               trainable=trainable))

    # 워드 임베딩 입력 → RNN → Dense(64) → 이진분류기
    model.add(layers.Bidirectional(layers.LSTM(64)))
    model.add(layers.Dense(64, activation='relu'))
    model.add(layers.Dense(1, activation='sigmoid'))

    model.compile(loss='binary_crossentropy',
                  optimizer='adam',
                  metrics=['accuracy'])
    return model


In [6]:
# 6. 모델 학습
def train_model(name, model):
    print(f"\n▶ Training: {name}")
    history = model.fit(train_pad, train_labels,
                        validation_data=(test_pad, test_labels),
                        epochs=5,
                        batch_size=32)
    return history

# GloVe 기반 모델 생성 (하나는 임베딩 고정, 하나는 미세조정)
glove_model_frozen = build_model(trainable=False)          # 사전학습된 임베딩 고정
glove_model_finetune = build_model(trainable=True)         # 사전학습된 임베딩 미세조정

# 모델 학습 실행
history_glove_frozen = train_model("GloVe (frozen)", glove_model_frozen)
history_glove_finetune = train_model("GloVe (fine-tune)", glove_model_finetune)


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(



▶ Training: GloVe (frozen)
Epoch 1/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 29s 31ms/step - accuracy: 0.7266 - loss: 0.5403 - val_accuracy: 0.8030 - val_loss: 0.4269
Epoch 2/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 20s 25ms/step - accuracy: 0.8305 - loss: 0.3843 - val_accuracy: 0.8468 - val_loss: 0.3508
Epoch 3/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 22s 28ms/step - accuracy: 0.8570 - loss: 0.3330 - val_accuracy: 0.8470 - val_loss: 0.3545
Epoch 4/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 20s 26ms/step - accuracy: 0.8728 - loss: 0.2996 - val_accuracy: 0.8675 - val_loss: 0.3131
Epoch 5/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 22s 28ms/step - accuracy: 0.8849 - loss: 0.2727 - val_accuracy: 0.8599 - val_loss: 0.3243

▶ Training: GloVe (fine-tune)
Epoch 1/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 24s 27ms/step - accuracy: 0.7307 - loss: 0.5169 - val_accuracy: 0.8654 - val_loss: 0.3240
Epoch 2/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 41s 27ms/step - accuracy: 0.9002 - loss: 0.2487 - val_accuracy: 0.8807 - val_loss: 0.2852
Epoch 3/5
782/782 ━━━━━━━━━━━━━━━━━━━━

In [7]:
# 7. 두 모델의 예측 결과 비교 테스트
print("""
📌아래의 입력창에 감성 분석할 문장을 입력하세요.
예시 문장 중 하나를 복사하여 붙여넣어도 됩니다.

I don't know how I feel about this movie.
The plot was okay, but the dialogue was terrible.
Great acting but the story was hard to follow.
Absolutely fantastic! I loved every moment.
What a waste of time. Completely boring.
""")

# 사용자 입력 기반 감성 분석 비교 함수 정의
def compare_models_on_input(model_dict):
    while True:
        sentence = input("\n🎤 감성 분석할 문장을 입력하세요 (종료: q 또는 quit):\n> ")
        if sentence.lower() in ['q', 'quit']:
            print("종료합니다.")
            break

        # 입력 문장 전처리 및 시퀀스 변환
        cleaned = clean_text(sentence)
        seq = tokenizer.texts_to_sequences([cleaned])
        pad = pad_sequences(seq, maxlen=MAX_LEN, padding='post')

        # 결과 출력
        print(f"\n[입력 문장]: {sentence}")
        for name, model in model_dict.items():
            prob = model.predict(pad, verbose=0)[0][0]
            label = "Positive 😊" if prob >= 0.5 else "Negative 😠"
            print(f"  ▶ {name:<16}: {label} ({prob:.4f})")

# 모델 딕셔너리 (각 모델이 학습되어 있어야 함)
model_dict = {
    "GloVe (frozen)": glove_model_frozen,
    "GloVe (fine-tune)": glove_model_finetune
}

# 사용자 입력을 받아 실시간 감성 분석 실행
compare_models_on_input(model_dict)



📌아래의 입력창에 감성 분석할 문장을 입력하세요.
예시 문장 중 하나를 복사하여 붙여넣어도 됩니다.

I don't know how I feel about this movie.
The plot was okay, but the dialogue was terrible.
Great acting but the story was hard to follow.
Absolutely fantastic! I loved every moment.
What a waste of time. Completely boring.


🎤 감성 분석할 문장을 입력하세요 (종료: q 또는 quit):
> The plot was okay, but the dialogue was terrible.

[입력 문장]: The plot was okay, but the dialogue was terrible.
  ▶ GloVe (frozen)  : Negative 😠 (0.0569)
  ▶ GloVe (fine-tune): Negative 😠 (0.0083)

🎤 감성 분석할 문장을 입력하세요 (종료: q 또는 quit):
> Great acting but the story was hard to follow.

[입력 문장]: Great acting but the story was hard to follow.
  ▶ GloVe (frozen)  : Positive 😊 (0.8149)
  ▶ GloVe (fine-tune): Negative 😠 (0.4657)

🎤 감성 분석할 문장을 입력하세요 (종료: q 또는 quit):
> I don't know how I feel about this movie.

[입력 문장]: I don't know how I feel about this movie.
  ▶ GloVe (frozen)  : Positive 😊 (0.7066)
  ▶ GloVe (fine-tune): Negative 😠 (0.1741)

🎤 감성 분석할 문장을 입력하세요 (종료: q 또는 quit):
